# HAM10000 بدقتها الأصلية — الطريق الصادق فوق سقف DermaMNIST

نموذج `derma` الحالي عند **0.8070**، والسقف المنشور لـDermaMNIST **~0.75–0.77**. تجاوزناه،
لكن الوصول لأرقام الديرموسكوبي المنشورة (**0.85+**) ما يجي من وصفة أذكى على نفس البيانات —
يجي من **ترك التصغير**. DermaMNIST هي HAM10000 منزّلة لـ224 بكسل كحد أقصى؛ الأصل **600×450**.

هذا **تغيير مصدر البيانات، مو تغيير المقياس** — شرعي تماماً، ولازم ينُشر كمهمة منفصلة.

---

## 🚩 وأهم من الدقة: مشكلة تسريب الآفات

HAM10000 فيها **10,015 صورة** لـ**~7,470 آفة فريدة** — يعني آلاف الآفات إلها **أكثر من صورة**
(زوايا/تكبيرات مختلفة لنفس الشامة).

DermaMNIST عدها **10,015 عيّنة** بالضبط (7007/1003/2005) — نفس عدد الصور. إذا التقسيم انعمل
**على مستوى الصورة** عشوائياً، فصور نفس الآفة تنوزّع بين التدريب والاختبار، والنموذج يتعرّف
على آفة شافها قبل — **تسريب يضخّم الرقم**.

**وفحص التسريب مالتنا ما يمسك هذا:** هو يقارن **بصمة البايتات** (`verify_retrain_gains.py`)،
وصورتان مختلفتان لنفس الآفة **بايتاتهن مختلفة** — فتعدّي من الفحص. طلعت `0/2005` وهذا صحيح
لكنه يجاوب على سؤال أضيق مما نحتاج.

**هذا الدفتر يقيس حجم المشكلة بدل ما يفترضها**، ويدرّب على تقسيم **مجمّع حسب الآفة**.

> ⚠️ **رقم التقسيم المجمّع مو قابل للمقارنة مع رقم MedMNIST.** التقسيم المجمّع **أصعب**
> بطبيعته. إذا نزل الرقم، هذا **مو تراجع** — هذا الرقم الحقيقي بدون تسريب. ينُنشر كصف
> منفصل بالجدول، وياه وسم التقسيم.

**الخطوات:** `Runtime → Change runtime type → GPU (T4)` ثم شغّل بالترتيب.

## ١) المكتبات + البطاقة

In [ ]:
!pip -q install timm scikit-learn pandas
import torch, torchvision, numpy as np, pandas as pd
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU (T4)"
p = torch.cuda.get_device_properties(0)
print("torch %s | GPU %s | %.1f GB" % (torch.__version__, p.name, p.total_memory / 1e9))

## ٢) جلب HAM10000

مسارَان. **دَتافيرس هارفارد** (الرسمي) ما يحتاج حساب — جرّبه أول.
إذا فشل، استخدم Kaggle (يحتاج ترفع `kaggle.json` من إعدادات حسابك).

In [ ]:
import os, glob, zipfile, subprocess, json
os.makedirs("/content/ham", exist_ok=True)
IMG_DIR = "/content/ham/images"
os.makedirs(IMG_DIR, exist_ok=True)

def n_images():
    return len(glob.glob(os.path.join(IMG_DIR, "*.jpg")))

SOURCE = None

# --- المسار أ: Harvard Dataverse (بدون حساب) ---------------------------------------------
if n_images() < 10000:
    try:
        DOI = "doi:10.7910/DVN/DBW86T"
        api = ("https://dataverse.harvard.edu/api/datasets/:persistentId/"
               "?persistentId=" + DOI)
        meta = json.loads(subprocess.run(["curl", "-sL", api],
                                         capture_output=True, text=True, timeout=180).stdout)
        files = meta["data"]["latestVersion"]["files"]
        print("dataverse files:", [f["dataFile"]["filename"] for f in files])
        for f in files:
            name, fid = f["dataFile"]["filename"], f["dataFile"]["id"]
            if not (name.endswith(".zip") or "metadata" in name.lower()):
                continue
            dst = "/content/ham/" + name
            if not os.path.exists(dst):
                print("downloading", name, "...")
                subprocess.run(["curl", "-sL", "-o", dst,
                                "https://dataverse.harvard.edu/api/access/datafile/%d" % fid],
                               timeout=1800)
            if name.endswith(".zip"):
                with zipfile.ZipFile(dst) as z:
                    z.extractall(IMG_DIR)
        # الصور ممكن تنفك داخل مجلد فرعي — سوّيها مسطّحة
        for p_ in glob.glob("/content/ham/images/**/*.jpg", recursive=True):
            if os.path.dirname(p_) != IMG_DIR:
                os.rename(p_, os.path.join(IMG_DIR, os.path.basename(p_)))
        if n_images() >= 10000:
            SOURCE = "harvard dataverse"
    except Exception as e:
        print("dataverse path failed:", type(e).__name__, e)

# --- المسار ب: Kaggle (يحتاج kaggle.json) ------------------------------------------------
if n_images() < 10000:
    print("\nارفع kaggle.json (Kaggle -> Settings -> Create New Token)")
    from google.colab import files as gfiles
    gfiles.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    os.replace("kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
    !pip -q install kaggle
    !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/ham --unzip
    for p_ in glob.glob("/content/ham/**/*.jpg", recursive=True):
        if os.path.dirname(p_) != IMG_DIR:
            os.rename(p_, os.path.join(IMG_DIR, os.path.basename(p_)))
    SOURCE = "kaggle"

print("\nimages found: %d   source: %s" % (n_images(), SOURCE))
assert n_images() >= 10000, "التحميل ما اكتمل — شوف الرسائل فوق"

## ٣) البيانات الوصفية + ترتيب أصناف مطابق للمشروع

In [ ]:
meta_path = None
for cand in glob.glob("/content/ham/**/*", recursive=True):
    b = os.path.basename(cand).lower()
    if "metadata" in b and (b.endswith(".csv") or b.endswith(".tab")):
        meta_path = cand; break
print("metadata:", meta_path)
sep = "\t" if meta_path.endswith(".tab") else ","
df = pd.read_csv(meta_path, sep=sep)
print(df.columns.tolist())

# ترتيب الأصناف مطابق لـ DermaMNIST (أبجدي حسب رمز dx) — حتى يبقى الـcheckpoint متوافق
DX = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
CLASSES = ["actinic keratoses and intraepithelial carcinoma", "basal cell carcinoma",
           "benign keratosis-like lesions", "dermatofibroma", "melanoma",
           "melanocytic nevi", "vascular lesions"]
BINARY_POSITIVE = [0, 1, 4]          # akiec, bcc, mel  = خبيث/ما قبل خبيث
df["label"] = df["dx"].map({d: i for i, d in enumerate(DX)})
df["path"] = df["image_id"].map(lambda x: os.path.join(IMG_DIR, x + ".jpg"))
df = df[df["path"].map(os.path.exists)].reset_index(drop=True)
print("\nrows=%d  images=%d  lesions=%d" % (len(df), df.image_id.nunique(), df.lesion_id.nunique()))
print(df["dx"].value_counts().to_string())

## ٤) 🔬 تدقيق التسريب — كم تكلّف القسمة على مستوى الصورة؟

هذي الخلية **تقيس** المشكلة بدل ما تفترضها: تسوّي تقسيماً عشوائياً على مستوى الصورة
بنفس نسب DermaMNIST (70/10/20)، وتعدّ كم صورة اختبار تشارك **نفس الآفة** مع التدريب.

In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit

multi = df.groupby("lesion_id").size()
print("lesions with >1 image: %d of %d  (%.1f%% of images are repeat shots)"
      % ((multi > 1).sum(), len(multi), 100 * (len(df) - len(multi)) / len(df)))

# تقسيم عشوائي على مستوى الصورة — نفس أسلوب DermaMNIST
tr_i, te_i = train_test_split(np.arange(len(df)), test_size=0.20,
                              stratify=df["label"], random_state=0)
leaked = df.iloc[te_i]["lesion_id"].isin(set(df.iloc[tr_i]["lesion_id"])).sum()
print("\nIMAGE-LEVEL split : %d/%d test images (%.1f%%) share a lesion with train  <-- LEAK"
      % (leaked, len(te_i), 100 * leaked / len(te_i)))

# تقسيم مجمّع على مستوى الآفة — النظيف
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=0)
tr_g, te_g = next(gss.split(df, groups=df["lesion_id"]))
leaked_g = df.iloc[te_g]["lesion_id"].isin(set(df.iloc[tr_g]["lesion_id"])).sum()
print("LESION-LEVEL split: %d/%d test images share a lesion with train  <-- clean"
      % (leaked_g, len(te_g)))

## ٥) الإعدادات + التقسيم المجمّع

`SPLIT="lesion"` هو الافتراضي والصحيح. `SPLIT="image"` موجود **فقط** لقياس فجوة التسريب —
لا تنشر رقمه كإنجاز.

In [ ]:
SPLIT     = "lesion"        # "lesion" (نظيف) | "image" (للمقارنة فقط)
BINARY    = False           # True = خبيث/حميد
SIZE      = 384             # 384 على T4 مع b0 مريحة؛ 448 تحتاج BATCH أصغر
BACKBONE  = "efficientnet_b0"   # efficientnet_b0 | convnext_tiny | resnet50
EPOCHS    = 20
WARMUP    = 2
BATCH     = 16
LR, HEAD_LR = 2e-4, 1e-3
PATIENCE, DROPOUT, LABEL_SMOOTH = 6, 0.4, 0.05

g = df["lesion_id"] if SPLIT == "lesion" else pd.Series(np.arange(len(df)))
sp1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=0)
tr_idx, te_idx = next(sp1.split(df, groups=g))
sp2 = GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=0)
tr2, va2 = next(sp2.split(df.iloc[tr_idx], groups=g.iloc[tr_idx]))
tr_idx, va_idx = tr_idx[tr2], tr_idx[va2]

d_tr, d_va, d_te = df.iloc[tr_idx], df.iloc[va_idx], df.iloc[te_idx]
y_tr, y_va, y_te = [d["label"].values.copy() for d in (d_tr, d_va, d_te)]
if BINARY:
    pos = set(BINARY_POSITIVE)
    y_tr, y_va, y_te = [np.array([1 if v in pos else 0 for v in y]) for y in (y_tr, y_va, y_te)]
classes = (["benign (bkl/df/nv/vasc)", "malignant or pre-malignant (akiec/bcc/mel)"]
           if BINARY else CLASSES)
n_cls = len(classes)
KEY = "derma_ham" + ("_bin" if BINARY else "") + ("" if SPLIT == "lesion" else "_imgsplit")
print("KEY=%s  split=%s  train=%d val=%d test=%d  classes=%d"
      % (KEY, SPLIT, len(d_tr), len(d_va), len(d_te), n_cls))
print("test balance:", np.bincount(y_te, minlength=n_cls).tolist())

## ٦) التحميل + التدعيم بالدقة الأصلية

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch.nn as nn

IM_MEAN, IM_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(SIZE, scale=(0.7, 1.0), ratio=(0.85, 1.18)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomApply([transforms.ColorJitter(0.25, 0.25, 0.15, 0.03)], p=0.7),
    transforms.RandomRotation(25),
    transforms.ToTensor(), transforms.Normalize(IM_MEAN, IM_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.12))])
eval_tf = transforms.Compose([
    transforms.Resize((SIZE, SIZE)), transforms.ToTensor(),
    transforms.Normalize(IM_MEAN, IM_STD)])

class HAM(Dataset):
    def __init__(self, d, y, tf): self.p = d["path"].values; self.y = y; self.tf = tf
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.tf(Image.open(self.p[i]).convert("RGB")), int(self.y[i])

def build_net(arch, num_classes, dropout):
    def head(f):
        return nn.Sequential(nn.Dropout(dropout), nn.Linear(f, num_classes))
    if arch == "efficientnet_b0":
        n = torchvision.models.efficientnet_b0(weights="IMAGENET1K_V1")
        n.classifier = head(n.classifier[1].in_features); return n, n.classifier
    if arch == "convnext_tiny":
        n = torchvision.models.convnext_tiny(weights="IMAGENET1K_V1")
        f = n.classifier[2].in_features; n.classifier[2] = head(f); return n, n.classifier[2]
    if arch == "resnet50":
        n = torchvision.models.resnet50(weights="IMAGENET1K_V2")
        n.fc = head(n.fc.in_features); return n, n.fc
    raise ValueError(arch)

print(BACKBONE, "ready |", SIZE, "px")

## ٧) التدريب — نفس وصفة المشروع (مرحلتان، اختيار على دقة التحقق)

In [ ]:
import time, copy
from sklearn.metrics import accuracy_score, balanced_accuracy_score

tl = DataLoader(HAM(d_tr, y_tr, train_tf), batch_size=BATCH, shuffle=True,
                num_workers=2, pin_memory=True)
vl = DataLoader(HAM(d_va, y_va, eval_tf), batch_size=BATCH * 2, num_workers=2, pin_memory=True)
el = DataLoader(HAM(d_te, y_te, eval_tf), batch_size=BATCH * 2, num_workers=2, pin_memory=True)

cnt = np.bincount(y_tr, minlength=n_cls)
w = cnt.sum() / (n_cls * np.maximum(cnt, 1))
net, head = build_net(BACKBONE, n_cls, DROPOUT); net = net.cuda()
crit = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).cuda(),
                           label_smoothing=LABEL_SMOOTH)
scaler = torch.amp.GradScaler("cuda", enabled=True)
head_ids = {id(p) for p in head.parameters()}
def freeze(fr):
    for p in net.parameters(): p.requires_grad = (id(p) in head_ids) if fr else True

@torch.no_grad()
def collect(loader, tta=False):
    net.eval(); ys, ps = [], []
    for xb, yb in loader:
        xb = xb.cuda(non_blocking=True)
        with torch.amp.autocast("cuda"):
            p = torch.softmax(net(xb), 1)
            if tta: p = (p + torch.softmax(net(torch.flip(xb, dims=[3])), 1)) / 2
        ps.append(p.float().cpu().numpy()); ys.append(yb.numpy())
    return np.concatenate(ys), np.concatenate(ps)

freeze(True)
opt = torch.optim.AdamW([p for p in net.parameters() if p.requires_grad],
                        lr=HEAD_LR, weight_decay=1e-4)
stage, sched, best, best_state, bad, hist = "A(head)", None, -1, None, 0, []
t0 = time.time()
for ep in range(EPOCHS):
    if ep == WARMUP:
        freeze(False)
        opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, max(1, EPOCHS - WARMUP))
        stage = "B(full)"
    net.train(); tot = 0.0
    for xb, yb in tl:
        xb, yb = xb.cuda(non_blocking=True), yb.cuda(non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            loss = crit(net(xb), yb)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item() * len(xb)
    if sched: sched.step()
    yv, pv = collect(vl); vacc = accuracy_score(yv, pv.argmax(1))
    hist.append({"epoch": ep + 1, "stage": stage, "train_loss": round(tot / len(y_tr), 4),
                 "val_acc": round(float(vacc), 4)})
    print("  ep %2d/%d [%s] loss=%.4f val_acc=%.4f val_bacc=%.4f"
          % (ep + 1, EPOCHS, stage, tot / len(y_tr), vacc,
             balanced_accuracy_score(yv, pv.argmax(1))), flush=True)
    if not np.isfinite(tot): raise SystemExit("[FATAL] non-finite loss — disable AMP")
    if vacc > best + 1e-4: best, best_state, bad = vacc, copy.deepcopy(net.state_dict()), 0
    else:
        bad += 1
        if bad >= PATIENCE: print("  [early stop]"); break
net.load_state_dict(best_state)
print("trained in %.0f s" % (time.time() - t0))

## ٨) التقييم — قرار TTA على val، والاختبار يُلمس مرّة وحدة

In [ ]:
from sklearn.metrics import (roc_auc_score, f1_score, confusion_matrix,
                             classification_report)
yv, pv_no = collect(vl); _, pv_tta = collect(vl, tta=True)
USE_TTA = accuracy_score(yv, pv_tta.argmax(1)) > accuracy_score(yv, pv_no.argmax(1))
print("val no-TTA=%.4f  TTA=%.4f  -> use_tta=%s"
      % (accuracy_score(yv, pv_no.argmax(1)), accuracy_score(yv, pv_tta.argmax(1)), USE_TTA))

yt, pt = collect(el, tta=USE_TTA)
pred = pt.argmax(1)
test_acc = accuracy_score(yt, pred)
print("\nTEST accuracy       = %.4f" % test_acc)
print("TEST balanced acc   = %.4f" % balanced_accuracy_score(yt, pred))
print("TEST macro F1       = %.4f" % f1_score(yt, pred, average="macro"))
print(classification_report(yt, pred, target_names=classes, zero_division=0))
try:
    ptn = pt / np.clip(pt.sum(1, keepdims=True), 1e-12, None)
    auc = (roc_auc_score(yt, ptn[:, 1]) if n_cls == 2
           else roc_auc_score(yt, ptn, multi_class="ovr", average="macro"))
except Exception as e:
    print("[warn] AUC:", e); auc = None
print("TEST AUC =", auc)
print("\nDermaMNIST (224px, image-level split) للمقارنة: 0.8070")
print("تذكير: تقسيم الآفة أصعب — الأرقام مو قابلة للمقارنة المباشرة.")

## ٩) الحفظ

In [ ]:
import shutil
os.makedirs("/content/out", exist_ok=True)
torch.save({"state_dict": net.state_dict(), "size": SIZE, "classes": classes,
            "mean": IM_MEAN, "std": IM_STD, "dropout": DROPOUT,
            "binary_task": BINARY, "tta": bool(USE_TTA), "arch": BACKBONE,
            "medmnist": None, "source_dataset": "HAM10000 (original resolution)",
            "split_type": SPLIT,
            "binary_positive": (BINARY_POSITIVE if BINARY else None)},
           "/content/out/%s.pt" % KEY)

json.dump({
    "model": "%s_%s" % (KEY, BACKBONE), "dataset_key": KEY,
    "source_dataset": "HAM10000 (original resolution, %d px)" % SIZE,
    "split_type": SPLIT, "backbone": BACKBONE, "input_size": SIZE,
    "binary_task": BINARY, "classes": classes, "n_classes": n_cls,
    "n_train": int(len(y_tr)), "n_val": int(len(y_va)), "n_test": int(len(y_te)),
    "test_accuracy": round(float(test_acc), 4),
    "test_balanced_accuracy": round(float(balanced_accuracy_score(yt, pred)), 4),
    "test_macro_f1": round(float(f1_score(yt, pred, average="macro")), 4),
    "test_auc": None if auc is None else round(float(auc), 4),
    "tta_used": bool(USE_TTA), "confusion_matrix": confusion_matrix(yt, pred).tolist(),
    "epoch_history": hist, "trained_on": "google colab",
    "NOT_COMPARABLE_TO": ("api/models/derma_v2_metrics.json — that model uses the official "
                          "DermaMNIST 224px IMAGE-level split. This run uses a %s-level split "
                          "at %d px. Publish as a separate row, never as an improvement on it."
                          % (SPLIT, SIZE)),
    "VERIFY_BEFORE_PUBLISHING": ("Re-measure locally before putting this in any table."),
}, open("/content/out/%s_metrics.json" % KEY, "w", encoding="utf-8"),
   ensure_ascii=False, indent=2)

shutil.make_archive("/content/%s" % KEY, "zip", "/content/out")
print("files:", os.listdir("/content/out"))
from google.colab import files as gfiles
gfiles.download("/content/%s.zip" % KEY)

## ١٠) التركيب والتسجيل

1. حط `<KEY>.pt` و`<KEY>_metrics.json` بـ`api/models/`.
2. `api/nets.py` يبني `resnet18` فقط — الجذوع الثانية تحتاج فرعاً؛ الخلية الأخيرة من
   `MedMNIST_Colab_Train.ipynb` تطبعه لك.
3. سجّل بـ`TRAINING_LOG.md` **كصف منفصل**، وياه:
   - `split_type` (lesion أو image)
   - الدقة بالبكسل
   - وجملة صريحة إنه **مو قابل للمقارنة** مع `derma_v2` (0.8070)

### 🔬 التجربة الي تستاهل تنعمل

شغّل الدفتر **مرّتين** — `SPLIT="lesion"` ثم `SPLIT="image"` — بنفس كل شي غيره.
الفرق بين الرقمين هو **قياس مباشر لكم يضخّم تسريب الآفات الرقم**. وهذا يجاوب على سؤال يخص
`derma_v2` نفسه، لأن DermaMNIST تستخدم التقسيم على مستوى الصورة.

إذا الفجوة كبيرة، فرقم `derma_v2 = 0.8070` **متفائل** — وهذا اكتشاف يستاهل التسجيل أكثر
من أي نقطة دقة نكسبها.